# Book Recommender — Phase 3: NLP, TF-IDF & K-Means Clustering

## What this notebook does

In Phase 2 we cleaned and explored our dataset. Now we answer a harder question:

> **How do you make a computer understand that 'Harry Potter' and 'The Hobbit' are similar books?**

Computers cannot read words — they only understand numbers. This phase converts book descriptions into numerical vectors using **TF-IDF**, then uses **K-Means clustering** to automatically group similar books together.

### Phases covered:
1. **Feature Engineering** — TF-IDF vectorisation of book content
2. **K-Means Clustering** — grouping books by similarity
3. **Elbow Method** — finding the optimal number of clusters
4. **PCA Visualisation** — seeing the clusters in 2D
5. **Cluster Analysis** — understanding what each cluster represents

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# Clustering
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

# Recommendation
from sklearn.metrics.pairwise import cosine_similarity

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.facecolor'] = '#FAFAFA'
plt.rcParams['figure.facecolor'] = 'white'

BLUE  = '#2C3E50'
RED   = '#E74C3C'
GREEN = '#27AE60'
AMBER = '#F39C12'
PALETTE = ['#2C3E50','#E74C3C','#27AE60','#F39C12','#8E44AD',
           '#16A085','#D35400','#2980B9','#C0392B','#1ABC9C',
           '#F39C12','#7F8C8D','#6C3483','#117A65','#B7950B']

print('Libraries loaded successfully')

In [ ]:
# Load clean dataset from Phase 2
df = pd.read_csv('clean_books.csv')
print(f'Dataset loaded: {len(df):,} books')
print(f'Columns: {df.columns.tolist()}')
print(f'\nSample content field:')
for i in range(3):
    print(f'  {df["content"].iloc[i][:80]}')

## 2. Feature Engineering — TF-IDF Vectorisation

### What is TF-IDF?

**TF-IDF** stands for **Term Frequency - Inverse Document Frequency**. It converts text into numbers in a smarter way than just counting words.

- **TF (Term Frequency)** — how often a word appears in one book's description
- **IDF (Inverse Document Frequency)** — how rare that word is across ALL books

**Why IDF matters:** The word 'fiction' appears in thousands of books — it tells us very little about any specific book. The word 'wizardry' appears in very few books — it's highly distinctive. TF-IDF gives high scores to distinctive words and low scores to common ones.

**Result:** Each book becomes a vector of numbers — a point in high-dimensional space where similar books are close together.

In [ ]:
# ── Step 1: Prepare the content field ─────────────────────────────────────

# Clean the content field
df['content'] = df['content'].fillna('')
df['content'] = df['content'].str.lower()          # lowercase everything
df['content'] = df['content'].str.replace(r'[^a-z\s]', ' ', regex=True)  # remove special chars
df['content'] = df['content'].str.replace(r'\s+', ' ', regex=True)       # normalise spaces

print('Content field cleaned')
print(f'Sample: {df["content"].iloc[0][:100]}')

In [ ]:
# ── Step 2: Build the TF-IDF matrix ───────────────────────────────────────
#
# Parameters explained:
#   max_features=5000  — keep only the 5,000 most informative terms
#   stop_words='english' — remove common words (the, a, is, in...)
#   ngram_range=(1,2)  — include single words AND two-word phrases
#                        e.g. 'science' AND 'science fiction' as separate features
#   min_df=2           — ignore terms that appear in fewer than 2 books
#   max_df=0.95        — ignore terms that appear in more than 95% of books

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

tfidf_matrix = tfidf.fit_transform(df['content'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]:,} books × {tfidf_matrix.shape[1]:,} features')
print(f'\nSample feature names (first 20):')
print(tfidf.get_feature_names_out()[:20].tolist())
print(f'\nSample feature names (last 20):')
print(tfidf.get_feature_names_out()[-20:].tolist())

In [ ]:
# ── Step 3: Visualise top TF-IDF terms for a sample book ──────────────────

def show_top_tfidf_terms(book_idx: int, n: int = 10):
    """Show the top TF-IDF scoring terms for a specific book."""
    feature_names = tfidf.get_feature_names_out()
    book_vector   = tfidf_matrix[book_idx].toarray()[0]
    top_indices   = book_vector.argsort()[::-1][:n]
    top_terms     = [(feature_names[i], book_vector[i]) for i in top_indices if book_vector[i] > 0]

    book_title = df['title'].iloc[book_idx]
    print(f'Book: "{book_title}"')
    print(f'Top {n} TF-IDF terms:')
    for term, score in top_terms:
        bar = '█' * int(score * 40)
        print(f'  {term:<30} {score:.4f} {bar}')

# Show for a few sample books
for idx in [0, 100, 500, 1000]:
    show_top_tfidf_terms(idx)
    print()

## 3. K-Means Clustering

### What is K-Means?

K-Means is an unsupervised learning algorithm that groups data points into **K clusters** based on similarity. It works by:

1. Randomly placing K 'centroids' (cluster centres) in the data
2. Assigning each book to its nearest centroid
3. Moving each centroid to the average position of its assigned books
4. Repeating steps 2-3 until the centroids stop moving

**For book recommendations:** Books in the same cluster are similar. When a user likes a book, we recommend other books from the same cluster.

**The key question: How many clusters (K) should we use?** We use the **Elbow Method** to find out.

In [ ]:
# ── Elbow Method ───────────────────────────────────────────────────────────
#
# We try K from 2 to 20 and measure the 'inertia' for each
# Inertia = total distance of all books from their cluster centre
# Lower inertia = tighter, better clusters
# The 'elbow' is where adding more clusters stops helping much

print('Running Elbow Method — testing K from 2 to 20...')
print('(This may take 1-2 minutes)')

inertias     = []
sil_scores   = []
K_range      = range(2, 21)

# Normalise matrix for better clustering
tfidf_norm = normalize(tfidf_matrix)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=100)
    km.fit(tfidf_norm)
    inertias.append(km.inertia_)
    sil = silhouette_score(tfidf_norm, km.labels_, sample_size=2000, random_state=42)
    sil_scores.append(sil)
    print(f'  K={k:2d} | Inertia: {km.inertia_:.1f} | Silhouette: {sil:.4f}')

print('\nDone!')

In [ ]:
# ── Plot Elbow Curve ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inertia (Elbow)
axes[0].plot(list(K_range), inertias, 'o-', color=BLUE,
             linewidth=2, markersize=6)
axes[0].set_title('Elbow Method — Inertia by K', fontsize=13,
                  fontweight='bold', pad=12)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

# Silhouette scores
best_k_idx = sil_scores.index(max(sil_scores))
best_k     = list(K_range)[best_k_idx]
axes[1].plot(list(K_range), sil_scores, 's-', color=GREEN,
             linewidth=2, markersize=6)
axes[1].axvline(best_k, color=RED, linestyle='--',
                linewidth=1.5, label=f'Best K={best_k}')
axes[1].set_title('Silhouette Score by K', fontsize=13,
                  fontweight='bold', pad=12)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Finding Optimal Number of Clusters',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBest K by silhouette score: {best_k}')
print(f'Silhouette score at K={best_k}: {max(sil_scores):.4f}')
print(f'\nA silhouette score of 0.0-0.25 = weak clusters (expected for text data)')
print(f'Text data naturally has overlapping clusters — this is normal.')

In [ ]:
# ── Choose K and fit final model ───────────────────────────────────────────
#
# We use K=10 as a balance between:
# - Enough clusters to meaningfully separate genres
# - Not so many that clusters become too small to be useful
# You can adjust this based on the elbow curve above

K = 10  # adjust based on your elbow curve

print(f'Fitting K-Means with K={K}...')

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10,
    max_iter=300
)
kmeans.fit(tfidf_norm)

# Add cluster labels to dataframe
df['cluster'] = kmeans.labels_

print(f'\nCluster sizes:')
cluster_sizes = df['cluster'].value_counts().sort_index()
for cluster, size in cluster_sizes.items():
    print(f'  Cluster {cluster}: {size:,} books')

print(f'\nAverage cluster size: {cluster_sizes.mean():.0f} books')

## 4. PCA Visualisation

### What is PCA?

Our TF-IDF matrix has 5,000 dimensions — one per feature. Humans cannot visualise 5,000 dimensions.

**PCA (Principal Component Analysis)** reduces those 5,000 dimensions down to 2, while preserving as much of the meaningful variation as possible. Think of it as "squashing" the data onto a 2D map where similar books are still close together.

This lets us actually **see** the clusters.

In [ ]:
# ── Reduce to 2D with PCA ──────────────────────────────────────────────────

print('Reducing dimensions with PCA...')
pca = PCA(n_components=2, random_state=42)

# Convert sparse matrix to dense for PCA
# Use a sample of 3000 books for speed — still representative
sample_size = min(3000, len(df))
sample_idx  = np.random.choice(len(df), sample_size, replace=False)

tfidf_sample   = tfidf_norm[sample_idx]
pca_coords     = pca.fit_transform(tfidf_sample.toarray())
sample_clusters = df['cluster'].iloc[sample_idx].values
sample_titles   = df['title'].iloc[sample_idx].values

explained = pca.explained_variance_ratio_
print(f'PCA variance explained: {explained[0]:.1%} + {explained[1]:.1%} = {sum(explained):.1%} total')
print(f'\nNote: Low explained variance is normal for text data with many features.')
print(f'The clusters are still meaningful in the original high-dimensional space.')

In [ ]:
# ── Plot PCA clusters ──────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 9))

for cluster_id in range(K):
    mask = sample_clusters == cluster_id
    ax.scatter(
        pca_coords[mask, 0],
        pca_coords[mask, 1],
        c=PALETTE[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.5,
        s=15,
        edgecolors='none'
    )

ax.set_title(f'Book Clusters — PCA Visualisation (K={K}, n={sample_size:,} books)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance explained)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance explained)')
ax.legend(loc='upper right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('chart_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEach dot is a book. Books of the same colour belong to the same cluster.')
print('Clusters that overlap in 2D may still be well-separated in 5,000 dimensions.')

## 5. Cluster Analysis — What Does Each Cluster Represent?

We can interpret each cluster by looking at:
1. The top TF-IDF terms (what words define this cluster)
2. Sample book titles from the cluster

In [ ]:
# ── Top terms per cluster ──────────────────────────────────────────────────

feature_names = tfidf.get_feature_names_out()
cluster_labels = {}

print(f'TOP TERMS AND SAMPLE BOOKS PER CLUSTER')
print('=' * 65)

for cluster_id in range(K):
    # Get top terms from cluster centroid
    centroid    = kmeans.cluster_centers_[cluster_id]
    top_indices = centroid.argsort()[::-1][:8]
    top_terms   = [feature_names[i] for i in top_indices]

    # Get sample books from this cluster
    cluster_books = df[df['cluster'] == cluster_id]['title'].head(5).tolist()
    cluster_size  = (df['cluster'] == cluster_id).sum()

    print(f'\nCluster {cluster_id} ({cluster_size:,} books)')
    print(f'  Top terms: {" | ".join(top_terms[:6])}')
    print(f'  Sample books:')
    for book in cluster_books:
        print(f'    - {book[:60]}')

    # Store label for chart
    cluster_labels[cluster_id] = top_terms[0].title()

In [ ]:
# ── Cluster size chart ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 5))

cluster_ids   = list(range(K))
cluster_sizes = [( df['cluster'] == c).sum() for c in cluster_ids]
cluster_lbls  = [f"Cluster {c}\n({cluster_labels.get(c, '')})"
                 for c in cluster_ids]

bars = ax.bar(cluster_lbls, cluster_sizes,
              color=PALETTE[:K], edgecolor='white', alpha=0.85)

for bar, val in zip(bars, cluster_sizes):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 20,
            f'{val:,}', ha='center', fontsize=9, fontweight='600')

ax.set_title(f'Books per Cluster (K={K})', fontsize=13,
             fontweight='bold', pad=12)
ax.set_ylabel('Number of Books')
ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
plt.savefig('chart_cluster_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNote: Unequal cluster sizes are normal for text clustering.')
print('Some themes (fiction) naturally have more books than others (poetry).')

## 6. Content-Based Recommendation Engine

Now we combine everything into a recommendation function that:
1. Finds the input book in our dataset
2. Identifies which cluster it belongs to
3. Uses cosine similarity within that cluster to find the most similar books

**Why cosine similarity?** It measures the angle between two vectors — books with similar TF-IDF term distributions will have a small angle (high cosine similarity) regardless of how long their descriptions are.

In [ ]:
# ── Build recommendation function ─────────────────────────────────────────

def recommend_books(title: str, n: int = 5, use_cluster: bool = True) -> pd.DataFrame:
    """
    Recommend books similar to the input title.

    Args:
        title:       Book title to find recommendations for
        n:           Number of recommendations to return
        use_cluster: If True, only compare within the same cluster (faster, more focused)
                     If False, compare against all books (slower, broader)

    Returns:
        DataFrame with recommended books and similarity scores
    """
    # Find the book
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if matches.empty:
        print(f'No book found matching "{title}"')
        return pd.DataFrame()

    # Use first match
    book     = matches.iloc[0]
    book_idx = matches.index[0]
    cluster  = book['cluster']

    print(f'Found: "{book["title"]}"')
    print(f'Author: {book["authors"]}')
    print(f'Genre: {book["genre"]} | Cluster: {cluster} ({cluster_labels.get(cluster, "")})')
    if pd.notna(book['avg_rating']):
        print(f'Rating: {book["avg_rating"]:.2f}/5.0 ({int(book["ratings_count"]):,} ratings)')
    print()

    # Get comparison pool
    if use_cluster:
        pool_idx = df[df['cluster'] == cluster].index.tolist()
        print(f'Searching within Cluster {cluster} ({len(pool_idx):,} books)...')
    else:
        pool_idx = df.index.tolist()
        print(f'Searching across all {len(pool_idx):,} books...')

    # Compute cosine similarity
    book_vector   = tfidf_norm[book_idx]
    pool_matrix   = tfidf_norm[pool_idx]
    similarities  = cosine_similarity(book_vector, pool_matrix).flatten()

    # Get top results (excluding the book itself)
    sim_series = pd.Series(similarities, index=pool_idx)
    sim_series = sim_series.drop(book_idx, errors='ignore')
    top_indices = sim_series.nlargest(n).index

    results = df.loc[top_indices, ['title', 'authors', 'genre', 'avg_rating', 'cluster']].copy()
    results['similarity'] = sim_series[top_indices].values
    results['similarity_pct'] = (results['similarity'] * 100).round(1)
    results = results.reset_index(drop=True)

    return results


print('Recommendation function ready.')
print('Usage: recommend_books("Harry Potter", n=5)')

In [ ]:
# ── Test the recommender ───────────────────────────────────────────────────

# Test 1 — Fantasy
print('=' * 55)
print('TEST 1 — Fantasy book')
print('=' * 55)
recs1 = recommend_books('Harry Potter', n=5)
if not recs1.empty:
    print('\nRecommendations:')
    print(recs1[['title', 'authors', 'similarity_pct']].to_string(index=False))

In [ ]:
# Test 2 — Nonfiction
print('=' * 55)
print('TEST 2 — Nonfiction book')
print('=' * 55)
recs2 = recommend_books('Sapiens', n=5)
if not recs2.empty:
    print('\nRecommendations:')
    print(recs2[['title', 'authors', 'similarity_pct']].to_string(index=False))

In [ ]:
# Test 3 — Your choice
print('=' * 55)
print('TEST 3 — Try your own book')
print('=' * 55)
# Change this to any book title in the dataset
my_book = 'The Great Gatsby'
recs3 = recommend_books(my_book, n=5)
if not recs3.empty:
    print('\nRecommendations:')
    print(recs3[['title', 'authors', 'similarity_pct']].to_string(index=False))

In [ ]:
# ── Compare cluster vs global recommendations ──────────────────────────────

print('CLUSTER vs GLOBAL — comparing recommendation approaches')
print('=' * 55)

print('\nCLUSTER-BASED (focused within cluster):')
r_cluster = recommend_books('Pride and Prejudice', n=5, use_cluster=True)
if not r_cluster.empty:
    print(r_cluster[['title', 'similarity_pct']].to_string(index=False))

print('\nGLOBAL (across all books):')
r_global = recommend_books('Pride and Prejudice', n=5, use_cluster=False)
if not r_global.empty:
    print(r_global[['title', 'similarity_pct']].to_string(index=False))

## 7. Save Model & Results

In [ ]:
import pickle

# Save the clustered dataset
df.to_csv('clustered_books.csv', index=False)

# Save the TF-IDF vectoriser and K-Means model
with open('tfidf_model.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

# Save the TF-IDF matrix
import scipy.sparse as sp
sp.save_npz('tfidf_matrix.npz', tfidf_norm)

print('Saved:')
print('  clustered_books.csv  — dataset with cluster labels')
print('  tfidf_model.pkl      — TF-IDF vectoriser')
print('  kmeans_model.pkl     — K-Means model')
print('  tfidf_matrix.npz     — TF-IDF matrix')
print(f'\nNext step: Phase 4 — Streamlit App')

## 8. Phase 3 Summary

| Step | What we did | Why |
|------|------------|-----|
| TF-IDF | Converted 11,223 book descriptions into 5,000-dimensional vectors | Computers need numbers not words |
| Elbow Method | Tested K from 2 to 20 to find optimal clusters | Data-driven decision on K |
| K-Means | Grouped all books into K clusters | Similar books land in same cluster |
| PCA | Reduced 5,000 dimensions to 2D | Visualise what the model learned |
| Recommender | Cosine similarity within cluster | Fast, focused recommendations |

**Next: Phase 4 — Build the Streamlit app that brings everything together into a user-facing interface.**